[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/NEURAL_BIZ/blob/master/Bishop_Ch_12/NB_bishop_ch12_finbert.ipynb)

# FinBERT: Sentiment Analysis for Earnings Calls

**Bishop, Chapter 12 -- Transformers**

This notebook demonstrates a practical financial NLP application of the Transformer architecture (Bishop Ch. 12): using **FinBERT**, a BERT model fine-tuned on financial text, to perform sentiment analysis on earnings call transcripts.

FinBERT was introduced by Araci (2019) and is based on the BERT encoder architecture covered in Chapter 12. It classifies financial text into **positive**, **negative**, or **neutral** sentiment -- a critical building block for quantitative trading signals.

**Quantlet**: [https://github.com/danpele/NEURAL_BIZ](https://github.com/danpele/NEURAL_BIZ)

## 1. Setup

In [ ]:
# !pip install transformers torch pandas numpy matplotlib -q

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Course color palette
BLUE = '#1f4e79'
CRIMSON = '#c00000'
GREEN = '#2e7d32'
GRAY = '#808080'

# Matplotlib defaults
matplotlib.rcParams['figure.facecolor'] = 'none'
matplotlib.rcParams['axes.facecolor'] = 'none'
matplotlib.rcParams['savefig.facecolor'] = 'none'
matplotlib.rcParams['savefig.transparent'] = True
matplotlib.rcParams['axes.grid'] = False
matplotlib.rcParams['legend.framealpha'] = 0.0

def save_fig(fig, name, chart_dir='../../charts'):
    for ax in fig.get_axes():
        ax.grid(False)
    fig.savefig(f'{chart_dir}/{name}.pdf', bbox_inches='tight', transparent=True)
    fig.savefig(f'{chart_dir}/{name}.png', bbox_inches='tight', transparent=True, dpi=150)
    print(f'Saved {chart_dir}/{name}.pdf and .png')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch: {torch.__version__}, Device: {device}')

## 2. Load FinBERT

[FinBERT](https://huggingface.co/ProsusAI/finbert) is a BERT model (110M parameters) fine-tuned on financial communication text. It outputs three sentiment classes: **positive**, **negative**, and **neutral**.

Under the hood, it uses the same encoder-only Transformer architecture from Bishop Ch. 12, Section 12.3 (BERT), with a classification head on top of the `[CLS]` token representation.

In [ ]:
model_name = "ProsusAI/finbert"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model=model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1
)

print(f"Model: {model_name}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Labels: {model.config.id2label}")

## 3. Sample Earnings Call Sentences

We create a set of realistic sentences that might appear in an earnings call transcript. These cover revenue, guidance, margins, cost pressures, and strategic outlook -- a deliberate mix of positive, negative, and neutral tone.

In [ ]:
earnings_sentences = [
    "Revenue exceeded expectations, rising 18 percent year over year to a record 4.2 billion dollars.",
    "We are raising our full-year guidance to reflect stronger than anticipated demand across all segments.",
    "Gross margins expanded by 320 basis points, driven by operational efficiencies and favorable product mix.",
    "Our cloud division delivered exceptional growth, with annual recurring revenue surpassing 1 billion dollars.",
    "We returned 2.3 billion dollars to shareholders through dividends and share repurchases this quarter.",
    "Supply chain disruptions and rising input costs compressed margins significantly during the quarter.",
    "We are lowering our forward guidance due to softening demand in the consumer segment.",
    "Operating expenses increased 12 percent, outpacing revenue growth and pressuring profitability.",
    "Total revenue for the quarter was 3.1 billion dollars, roughly in line with consensus estimates.",
    "We maintained our dividend at 0.85 dollars per share, consistent with our capital allocation framework.",
]

print(f"Number of sentences: {len(earnings_sentences)}")
for i, s in enumerate(earnings_sentences):
    print(f"  [{i+1}] {s[:80]}...")

## 4. Run Sentiment Analysis

We pass each sentence through the FinBERT pipeline, which tokenizes the text, runs it through the Transformer encoder, and applies the classification head to produce sentiment probabilities.

In [ ]:
results = sentiment_pipeline(earnings_sentences)

df = pd.DataFrame({
    'text': [s[:75] + '...' if len(s) > 75 else s for s in earnings_sentences],
    'sentiment': [r['label'] for r in results],
    'confidence': [round(r['score'], 4) for r in results],
})

# Display the full DataFrame
pd.set_option('display.max_colwidth', 80)
print(df.to_string(index=True))
print(f"\nSentiment distribution:\n{df['sentiment'].value_counts().to_string()}")

## 5. Visualize Sentiment Results

A horizontal bar chart shows the confidence score for each sentence, color-coded by predicted sentiment: green for positive, red for negative, gray for neutral.

In [ ]:
color_map = {'positive': GREEN, 'negative': CRIMSON, 'neutral': GRAY}
colors = [color_map.get(s, GRAY) for s in df['sentiment']]

fig, ax = plt.subplots(figsize=(10, 6))

y_pos = range(len(df))
bars = ax.barh(y_pos, df['confidence'], color=colors, edgecolor='white', height=0.7)

ax.set_yticks(y_pos)
ax.set_yticklabels([f"[{i+1}]" for i in range(len(df))], fontsize=10)
ax.set_xlabel('Confidence Score', fontsize=12)
ax.set_title('FinBERT Sentiment Analysis -- Earnings Call Sentences', fontweight='bold', fontsize=13)
ax.set_xlim(0, 1.05)
ax.invert_yaxis()

# Add sentiment labels on bars
for i, (conf, sent) in enumerate(zip(df['confidence'], df['sentiment'])):
    ax.text(conf + 0.02, i, f'{sent} ({conf:.2f})', va='center', fontsize=9,
            color=color_map.get(sent, GRAY), fontweight='bold')

# Legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=GREEN, label='Positive'),
                   Patch(facecolor=CRIMSON, label='Negative'),
                   Patch(facecolor=GRAY, label='Neutral')]
ax.legend(handles=legend_elements, loc='lower right', fontsize=10)

fig.tight_layout()
save_fig(fig, 'bishop_ch12_finbert_sentiment')
plt.show()

## 6. Sentiment Score Distribution

We can also extract the full probability distribution over all three classes for each sentence, giving a richer view of the model's uncertainty.

In [ ]:
# Get full probability distributions using the model directly
all_probs = []
with torch.no_grad():
    for sentence in earnings_sentences:
        inputs = tokenizer(sentence, return_tensors="pt", padding=True, truncation=True, max_length=512)
        if torch.cuda.is_available():
            inputs = {k: v.to(device) for k, v in inputs.items()}
            model.to(device)
        outputs = model(**inputs)
        probs = torch.nn.functional.softmax(outputs.logits, dim=-1).cpu().numpy()[0]
        all_probs.append(probs)

probs_array = np.array(all_probs)
labels_order = [model.config.id2label[i] for i in range(3)]

# Stacked bar chart
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(earnings_sentences))
width = 0.65

label_colors = {'positive': GREEN, 'negative': CRIMSON, 'neutral': GRAY}
bottom = np.zeros(len(earnings_sentences))

for i, label in enumerate(labels_order):
    color = label_colors.get(label, BLUE)
    ax.barh(x, probs_array[:, i], left=bottom, height=width,
            label=label.capitalize(), color=color, alpha=0.85)
    bottom += probs_array[:, i]

ax.set_yticks(x)
ax.set_yticklabels([f"[{i+1}]" for i in range(len(earnings_sentences))], fontsize=10)
ax.set_xlabel('Probability', fontsize=12)
ax.set_title('FinBERT Full Probability Distribution per Sentence', fontweight='bold', fontsize=13)
ax.set_xlim(0, 1.0)
ax.invert_yaxis()
ax.legend(loc='lower right', fontsize=10)

fig.tight_layout()
save_fig(fig, 'bishop_ch12_finbert_probs')
plt.show()

## 7. Financial Application: Earnings Call Sentiment as a Trading Signal

### From NLP to Alpha

In quantitative finance, FinBERT sentiment scores from earnings calls can be used to construct **trading signals**:

1. **Transcript ingestion**: Parse earnings call transcripts (e.g., from SEC EDGAR or data vendors) into individual sentences or paragraphs.

2. **Sentence-level scoring**: Apply FinBERT to each sentence. Compute a numerical sentiment score, for example:
$$\text{score}_i = P(\text{positive})_i - P(\text{negative})_i \in [-1, +1]$$

3. **Aggregation**: Average sentence-level scores across the full transcript to get an **overall call sentiment**:
$$S_{\text{call}} = \frac{1}{N} \sum_{i=1}^{N} \text{score}_i$$

4. **Signal construction**: 
   - **Long** stocks with $S_{\text{call}} > \tau$ (bullish earnings tone)
   - **Short** stocks with $S_{\text{call}} < -\tau$ (bearish earnings tone)
   - Threshold $\tau$ is calibrated on historical data

5. **Backtesting**: Evaluate the strategy on historical data, measuring Sharpe ratio, maximum drawdown, and turnover. Academic studies (e.g., Araci 2019, Huang et al. 2020) report that FinBERT-based signals carry predictive power for short-term post-earnings returns.

### Example: Aggregate Sentiment Score

In [ ]:
# Compute numerical sentiment scores: P(positive) - P(negative)
pos_idx = list(model.config.id2label.values()).index('positive')
neg_idx = list(model.config.id2label.values()).index('negative')

sentence_scores = probs_array[:, pos_idx] - probs_array[:, neg_idx]

df['score'] = sentence_scores.round(4)
aggregate_score = sentence_scores.mean()

print("Sentence-level sentiment scores [P(pos) - P(neg)]:")
for i, (s, sc) in enumerate(zip(earnings_sentences, sentence_scores)):
    direction = "BULLISH" if sc > 0.1 else ("BEARISH" if sc < -0.1 else "NEUTRAL")
    print(f"  [{i+1}] {sc:+.3f}  {direction}")

print(f"\nAggregate call sentiment: {aggregate_score:+.3f}")
if aggregate_score > 0.1:
    print("Signal: LONG (bullish earnings tone)")
elif aggregate_score < -0.1:
    print("Signal: SHORT (bearish earnings tone)")
else:
    print("Signal: HOLD (mixed/neutral tone)")

In [ ]:
# Visualize sentence-level scores as a diverging bar chart
fig, ax = plt.subplots(figsize=(10, 6))

bar_colors = [GREEN if s > 0.1 else (CRIMSON if s < -0.1 else GRAY) for s in sentence_scores]
y_pos = np.arange(len(sentence_scores))

ax.barh(y_pos, sentence_scores, color=bar_colors, edgecolor='white', height=0.7)
ax.axvline(x=0, color='black', linewidth=0.8, linestyle='-')
ax.axvline(x=0.1, color=GREEN, linewidth=0.8, linestyle='--', alpha=0.5)
ax.axvline(x=-0.1, color=CRIMSON, linewidth=0.8, linestyle='--', alpha=0.5)

ax.set_yticks(y_pos)
ax.set_yticklabels([f"[{i+1}]" for i in range(len(sentence_scores))], fontsize=10)
ax.set_xlabel('Sentiment Score [P(pos) - P(neg)]', fontsize=12)
ax.set_title('FinBERT Trading Signal -- Sentence-Level Scores', fontweight='bold', fontsize=13)
ax.invert_yaxis()

# Annotate aggregate
ax.axvline(x=aggregate_score, color=BLUE, linewidth=2, linestyle='-', alpha=0.8)
ax.text(aggregate_score, len(sentence_scores) - 0.3,
        f'  Aggregate: {aggregate_score:+.2f}', color=BLUE, fontweight='bold', fontsize=10)

fig.tight_layout()
save_fig(fig, 'bishop_ch12_finbert_signal')
plt.show()

## 8. EU AI Act: Transparency Requirements for NLP in Finance

The **EU AI Act** (Regulation 2024/1689) classifies AI systems by risk level. Financial applications of NLP -- including automated sentiment analysis used for trading decisions or credit assessments -- may fall under **high-risk** or at minimum require transparency obligations:

- **Transparency (Article 50)**: Users interacting with AI-generated content or AI-driven decisions must be informed that AI is involved. If FinBERT sentiment scores feed into automated trading systems, the use of AI must be disclosed to relevant stakeholders.

- **High-risk classification (Annex III)**: AI systems used in creditworthiness assessment or credit scoring are explicitly listed as high-risk. If NLP-derived sentiment signals influence lending or investment decisions at scale, they may trigger conformity assessment requirements.

- **Human oversight (Article 14)**: High-risk systems must be designed to allow effective human oversight. A fully automated trading strategy driven by FinBERT would need mechanisms for human review, intervention, and override.

- **Data governance (Article 10)**: Training data for financial NLP models must be relevant, representative, and free of biases. FinBERT's training corpus (financial news and SEC filings) should be documented and auditable.

- **Risk management (Article 9)**: Firms deploying NLP-based trading signals must maintain a risk management system that identifies and mitigates risks throughout the AI system's lifecycle, including model drift in sentiment classification.

**Practical implication**: Financial institutions deploying FinBERT or similar models in the EU must maintain model cards, conduct bias audits, and ensure that human analysts can inspect and override model-driven signals.

## 9. Key Takeaways

In [ ]:
takeaways = [
    "1. FinBERT applies the BERT encoder architecture (Bishop Ch. 12) to financial text,\n"
    "   classifying sentences as positive, negative, or neutral with calibrated probabilities.",
    "2. Pre-trained Transformer models can be fine-tuned on domain-specific corpora (financial\n"
    "   news, SEC filings) to achieve strong performance on specialized NLP tasks.",
    "3. Sentence-level sentiment scores can be aggregated into a single call-level signal,\n"
    "   which serves as input to systematic long/short trading strategies.",
    "4. The full probability distribution (not just the argmax label) provides a richer signal:\n"
    "   high-confidence predictions carry more information than low-confidence ones.",
    "5. Under the EU AI Act, deploying NLP models for financial decision-making requires\n"
    "   transparency, human oversight, and documented risk management procedures.",
]

print("Key Takeaways:")
print("=" * 80)
for t in takeaways:
    print(f"\n{t}")